In [ ]:
import vllm

In [ ]:
from vllm import LLM, SamplingParams
import torch
import sys, os

# MPS 체크
use_mps = torch.backends.mps.is_available()
device_type = "mps" if use_mps else "cpu"

print(f"Using device: {device_type}")

# 작은 모델 사용 권장 (예: TinyLlama)
llm = LLM(
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    download_dir="./models",
    tensor_parallel_size=1,
    trust_remote_code=True,
    dtype="float16" if use_mps else "float32"
)

sampling_params = SamplingParams(temperature=0.7, top_p=0.95, max_tokens=100)

prompt = "Write a short poem about artificial intelligence."
outputs = llm.generate([prompt], sampling_params)

for output in outputs:
    print(output.outputs[0].text)


In [ ]:
for output in outputs:
    print(output.outputs[0].text)

In [ ]:
VLLM_USE_CUDA=0 python -m vllm.entrypoints.api_server \
  --model TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
  --tensor-parallel-size 1 \
  --host 127.0.0.1 \
  --port 8000 \
  --dtype float16


In [ ]:
import requests

url = "http://localhost:8000/v1/chat/completions"

payload = {
    "model": "kfkas/Legal-Llama-2-ko-7b-Chat",
    "messages": [
        {"role": "user", "content": "도로교통법 제 11조가 무슨 내용이야?"}
    ],
    "max_tokens": 100,
    "temperature": 0.7,
}

# --api-key 옵션 안 썼으면 Authorization 헤더는 없어도 됨
response = requests.post(url, json=payload)
print(response.status_code)
print(response.json())

In [ ]:
print(response.json()["choices"][0]["message"]['content'])

In [ ]:
import requests

url = "http://localhost:8000/v1/completions"

system_prompt = "너는 대한민국 현행 법령을 이해하고 설명하는 전문 법률 AI 비서다."
user_prompt = "도로교통법 제 11조가 무슨 내용이야?"

# LLaMA2 스타일 프롬프트 직접 구성
prompt = f"<s>[INST] <<SYS>>\n{system_prompt}\n<</SYS>>\n{user_prompt} [/INST]"

payload = {
    "model": "kfkas/Legal-Llama-2-ko-7b-Chat",
    "prompt": prompt,
    "max_tokens": 200,
    "temperature": 0.7,
}

response = requests.post(url, json=payload)
print("Status:", response.status_code)
print(response.json()["choices"][0]["text"])

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "kfkas/Legal-Llama-2-ko-7b-Chat"

# 1) 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2) 모델 로드
# Mac이면 dtype 적당히 float16 아니면 auto
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.backends.mps.is_available() else torch.float32,
    device_map="auto",
)

# 3) 사용자 입력
prompt = "대한민국 헌법 제10조 내용을 요약해줘."

# 4) 모델 입력 변환 (tokenize)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# 5) 모델 추론
output_ids = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
)

# 6) 디코딩(문자열로 변환)
response = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print(response)